# Relatório da Solução

## 1. Resumo do Contexto

Esse projeto tinha como objetivo criar um modelo preditivo capaz de fazer predições de temperaturas do aço contído em um Forno Panela durante o processo de Refino Secundário do aço da planta siderúrgica Steelproof. A planta tem interesse em otimizar seus processos e futuramente ser capaz de reduzir custos de energia elétrica devido ao aquecimento do forno panela. O processo de Refino Secundário é complexo e muitas vezes rico em decisões operacionais. Por esse motivo, é comum que medições de temperatura precisem ser repetidas ou que ciclos de aquecimento do aço acabem superaquecendo ou subaquecendo o material.

O objetivo ideal da aplicação da primeira versão do modelo proposto nesse projetos seria como um Sensor Virtual, capaz de atuar como referência preditiva de temperaturas do aço de acordo com as variáveis recentes do Lote de aço sendo refinado, como temperatura anterior, duração dos ciclos de aquecimento, quantidade e natureza dos materiais adicionados, etc.

O projeto demonstrou a viabilidade de desenvolver um modelo preditivo para estimar a temperatura do aço a partir dos registros operacionais disponíveis. O melhor modelo, CatBoost, obteve EAM de 3,93°C no conjunto de teste, reduzindo o erro em aproximadamente 59,25% em relação ao modelo de Persistência. Entretanto, os dados disponíveis não permitem estabelecer uma relação quantitativa entre as previsões do modelo e a redução dos custos de energia elétrica. Dessa forma, os resultados representam uma primeira etapa promissora para o desenvolvimento de um sensor virtual, mas sua aplicação operacional e seu impacto econômico ainda precisam ser validados.

## 2. Execução do Plano e Dificuldades

Os seguintes passos foram executados de acordo com o plano do experimento:

- Processamento e Limpeza de Dados:
  - Verificação e tratamento dos Valores ausentes
  - Identificadas e tratadas as observações sem temperaturas iniciais válidas para treinamento
  - Investigação e tratamento de valores extremos
  - Verificação de dados duplicados, integridade das chaves de identificação dos dados e consistência das chaves nos diversos conjuntos de dados
- Preparação de Dados:
  - Identificação dos eventos que correspondem a cada Intervalo a ser usado para treinamento do modelo (ciclos de aquecimento, adições de materiais)
  - Identificação dos intervalos válidos para treinamento do modelo
  - Agregamento dos dados de acordo com os Lotes e Intervalos de cada evento registrado nos dados
  - Escalonamento dos dados para modelo Ridge
  - Engenharia de Características para criar variáveis de treinamento como médias descritivas, intervalos temporais, durações, etc.
  - Particionamento dos dados de acordo com seus Lotes, chaves e respeitando a cronologia dos conjuntos de dados.
- Desenvolvimento de modelos seguindo os padrões estipulados (validação cruzada e ajuste de hiperparâmetros)
- Comunicação dos Resultados:
  - Vistualiação da Distribuição das predições em relação aos valores reais de temperatura, distribuição dos erros de predição, maiores erros, etc.
  - Relatório Final (esse documento)

Os seguintes passos não foram executados:
  - Exemplo de uso do modelo

A demonstração de uso do modelo não foi incluída como etapa independente, pois o processo de treinamento e avaliação já apresenta exemplos de predições sobre dados não utilizados no ajuste. Entretanto, esses exemplos não validam o uso do sensor virtual em instantes arbitrários durante o processo. Essa aplicação exigiria uma avaliação adicional com dados operacionais disponíveis em tempo real e medições de referência.

Em relação as dificuldades, a maioria tem relação direta com a limpeza dos dados e o agregamento de dados separados em um total de sete conjuntos de dados. O maior desafio foi conseguir construir conjuntos que alinhavam as características relacionadas a cada observação que seria usada para treinamento. Por fim, fazer o dseign das comparaçÕes dos modelos também foi um desafio oneroso mas não necessariamente denso em conteúdo. Para resolver ambos, resolvi fazer o design usando o resultado que gostaria de obter ao final da preparação dos dados e então construir o caminho até chegar lá. Primeiramente defini como seria o conjunto de dados de treinamento e então defini quais tipos de dados eu precisaria construir para cada um dos conjuntos de dados de maneira que eu conseguisse extrair o que era necessário de cada um. Isso resultou no uso de muitos conjuntos de dados de formato "long" e também de reindexamento, além de criação de novas chaves unívocas para identificar os diferentes tipos de dados.

## 3. Dados e Metodologia

Os dados recebidos são de boa qualidade mas tem limitações. Recebemos 7 conjuntos de dados com dados dos ciclos de aquecimento (incluindo potências reais e reativas, além de durações), medições de temperatura, purga de gás realizada nos lotes de Refino Secundário e adições de materiais (a granel e em arame) aos lotes. Os dados representam, em sua totalidade, 14876 ciclos de aquecimento de 3214 lotes de aço, 3219 lotes com adições de pelo menos um material a granel, 3239 lotes com purga de gás realizada, 15907 medições de temperatura de 3216 lotes de aço e 3081 lotes com adições de pelo menos um material em arame.

Com esses dados, definimos um "Intervalo" como todos os eventos que acontecem entre uma temperatura inicial $T_{1}$ e uma temperatura final $T_{2}$, de maneira que nosso modelo aprenda com as informações contidas nesse intervalo para finalmente poder prever $T_{2}$. Identificamos um total de 7368 Intervalos válidos para treinamento e teste do nosso modelo. Tivemos que remover 2901 medições de temperatura com valores ausentes, algumas medições de temperatura com valores muito baixos que suspeitamos e argumentos serem possíveis erros de medição e também precisamos não considerar alguns dados. Os dados de purga de gás não puderam ser utilizados por serem valores agregados de um Lote inteiro de Refino Secundário. Como nosso modelo atua em Intervalos e possam haver multiplos Intervalos por Lote, não foi possível identificar maneiras de aplicar os dados agregados de gás para cada Intervalo. Precisariamos de dados temporais de quando o gás foi purgado no Lote.

Por outro lado, também não temos informações sobre a natureza dos dados de energia elétrica. Embora saibamos que as colunas fazem referência a Potências, não sabemos as unidades de medida exatas que foram usadas e nem como essas colunas foram registradas. Não podemos assumir uma posição em relação a esses dados porque a maneira de utilizá-los seria diferente dependendo de sua natureza, como por exemplo se os mesmos são em si médias de consumo para cada ciclo de aquecimento ou totais nominais ou até mesmo configurações do maquinário utilizado. Não removemos os dados de energia elétrica, mas tivemos que utilizá-los como características descritivas simples e não como valores com relação direta com cada Intervalo. 

De todos os dados identificados, separamos 20% para teste e utilizamos 80% durante a fazer de treinamento e validação.

Foram definidos limiares de desempenho para avaliar os modelos desenvolvidos: EAM de até 10°C, REQM de até 15°C e melhoria relativa mínima de 20% em comparação com o modelo de Persistência. Esses limiares representam critérios de avaliação do projeto e não substituem as tolerâncias operacionais que devem ser estabelecidas pela Steelproof.

Os seguintes modelos foram identificados para treinamento:

- Dummy (modelo ingênuo que prevê a média)
- Persistência - modelo constante que prevê a última temperatura registrada
- Ridge
- RandomForestRegressor
- LightGBM
- CatBoost

Os dados foram escalonados e imputados para o treinamento de Ridge. Para RandomForestRegressor, fizemos o uso de imputação por estarmos usando a versão 0.24.1 de Scikit-Learn em que RandomForestRegressor não pode receber valores nulos. LightGBM e CatBoost não receberam tratamento especial nos dados.

Finalmente, executamos um treinamento usando Validação Cruzada para todos os modelos e seguimos com um segundo treinamento em que fizemos uma otimização de hiperparâmetros.

Padronizamos as comparações tanto em valores quanto em visualizações para que nossa análise fosse conveniente para o leitor.

## 4. Resultados e Desempenho

Performance de EAM de cada Modelo após Validação Cruzada e Otimização de Hiperparâmetros:


- CatBoost: 4.042
- LightGBM: 4.183
- Ridge: 4.295
- RandomForest: 4.416
- Persistence (baseline): 8.959
- Dummy (baseline): 11.794

O melhor modelo treinado foi CatBoost após uma otimizações de hiperparâmetros que resultou em EAM de $4.042^oC$ e REQM de $5.824^oC$ nos dados de treinamento. Nos dados reservados para teste, esse mesmo modelo obteve EAM $3.927^oC$ e REQM $5.584^oC$. O modelo ficou bem abaixo dos nossos KPIs de $10^oC$ e $15^oC$ para EAM e REQM respectivamente. No KPI de Melhora comparada com Baseline, nosso modelo também se mostrou eficaz, com uma melhora de 54.88% em dados de Treinamento e 59.25% em dados de teste. Os hiperparâmetros que encontramos como melhores foram 900 iterações, depth = 4 e learning_rate = 0.05.

Ainda assim, houveram limitações. Nosso modelo tem alguns erros muito altos para algumas de suas predições, embora essas não sejam tão comuns. Os erros tem a tendência a serem maiores para durações maiores de intervalos, mas intervalos mais longos que 30 minutos não são comuns em nosso conjunto de dados então essa é uma limitação estrutural do nosso problema. Os erros também são mais concentrados para os lotes com ciclos de aquecimento se comparados aos Intervalos que tem adições de materiais mas não tem ciclos de aquecimento. Esses Intervalos também são menos comuns.

Como etapa inicial de validação operacional, sugere-se priorizar intervalos com ciclos de aquecimento e duração de até 30 minutos, por serem condições mais representadas nos dados. Essa seleção não garante a confiabilidade das previsões e deverá ser acompanhada de medições diretas para avaliação de desempenho.

Por fim, a performance precisa ser avaliada por especialistas do campo para entender se existem riscos e limitações da operacionalização de tal Sensor Virtual como eventual substituto a medições manuais uma vez que é preciso definir as limitações metalúrgicas explicitamente para definir se o desempenho do modelo está adequado.

## 5. Impacto no Negócio

O impacto econômico potencial pode ser dividido em duas frentes principais: redução dos custos associados às medições diretas de temperatura e melhoria da eficiência energética do refino secundário. No primeiro caso, seria necessário identificar o custo unitário e a frequência das medições que poderiam ser substituídas com segurança. No segundo, seria necessário verificar se a disponibilidade das estimativas permite reduzir reaquecimentos, consumo elétrico ou tempo de processamento. Nenhum desses benefícios foi quantificado neste experimento, de modo que representam oportunidades a serem avaliadas, e não economias já demonstradas.

Como exploramos anteriornmente, o impacto do modelo de Sensor Virtual não pode ser quantificado diretamente como uma melhora no custo de energia usando somente os dados que a Steelproof disponibilizou. Ainda assim, conseguimos um modelo que tem performance preditiva bastante adequada se considerarmos que essa é uma primeira versão do modelo que teve que lidar com diversas limitações nos dados. Como exploramos no plano do projeto, outros modelos mais robustos e treinados em contextos acadêmicos conseguiram peformances muito interessantes por volta de 6C de EAM, embora em outros contextos e com dados mais ricos, muitas vezes prevendo a temperatura final do Lote inteiro. Dessa forma, é considerável que o impacto futuro do modelo tenha a tendência a se acentuar.

Um futuro modelo poderia utilizar os dados de energia elétrica e purga de gás de maneira direta, o que poderia enriquecuer a maneira que o modelo compreende as relações das diversas variáveis dos nossos dados. Poderíamos, em futuras versões do modelo, otimizar o experimento de forma que seja possível validar com mais confiabilidade de que um modelo preditivo que age como um Sensor Virtual possa substituir medições manuais. Naturalmente, conforme esse modelo melhora e sua performance se torna cada vez mais adequada para a substituição de medições manuais, a otimização do processo também aumenta. As medições manuais de temperatura resultam em paradas nos ciclos de Refino Secundário, muitas vezes gastanto energia por requerer seguintes reaquecimentos dos Lotes. Os processos também nem sempre atingem as temperaturas desejadas mesmo com bom planejamento dos intervalos. Um modelo preditivo robusto poderia futuramente sugerir características para um ciclo de aquecimento que venham a trazer a temperatura ideal e assim manter o desperdício de energia elétrica por reaquecimentos a um mínimo. 

Como discutimos em nosso plano, essas futuras versões irão requerer mais disponibilidade de dados e informações sobre os dados.

## 6. Conclusões e Recomendações

Os resultados indicam que o CatBoost é um candidato promissor para o desenvolvimento de um sensor virtual de temperatura, tendo superado os limiares de desempenho estabelecidos no projeto e reduzido significativamente o erro em relação ao modelo de Persistência.

Recomenda-se como próximo passo realizar uma validação operacional supervisionada, mantendo as medições diretas de temperatura e comparando-as com as estimativas do modelo. Essa etapa deve envolver as equipes de operação e controle de qualidade para definir as tolerâncias aceitáveis, identificar condições de maior risco e avaliar a possibilidade de utilização do sensor virtual como ferramenta de apoio.

Paralelamente, recomenda-se melhorar a coleta de dados, especialmente informações temporais sobre a purga de gás, características dos aços processados e registros mais detalhados de consumo elétrico. Esses dados poderiam contribuir tanto para o aperfeiçoamento das estimativas quanto para estudos futuros de otimização energética.